In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

import cv2 as cv
import numpy as np
import time
import csv
from tqdm import tqdm
from mtcnn import MTCNN

# Load MTCNN
print("Loading MTCNN Model...")
detector = MTCNN()

print("MTCNN Loaded Successfully!\n")

# Dataset Folder
image_folder = "/kaggle/input/datasets/ashwingupta3012/human-faces/Humans"

if not os.path.exists(image_folder):
    raise FileNotFoundError(f"{image_folder} folder not found.")

image_files = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print(f"Found {len(image_files)} images.\n")

# CSV Output
csv_file = "MTCNN_Evaluation_Results.csv"

total_faces = 0
total_latency = 0

with open(csv_file, "w", newline="", encoding="utf-8") as file:

    writer = csv.writer(file)

    writer.writerow([
        "Image Name",
        "Faces Detected",
        "Latency (ms)"
    ])

    for img_name in tqdm(image_files):

        img_path = os.path.join(image_folder, img_name)

        img = cv.imread(img_path)
        
        #Reduce image size to prevent memory crashes and speed up processing
        img = cv.resize(img, (640, 640))

        if img is None:
            continue

        rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)

        start = time.perf_counter()

        faces = detector.detect_faces(rgb)

        latency = (time.perf_counter() - start) * 1000

        face_count = len(faces)

        writer.writerow([
            img_name,
            face_count,
            round(latency,2)
        ])

        total_faces += face_count
        total_latency += latency
        
        del img
        del rgb
        del faces
        import gc
        gc.collect()

# Summary
avg_latency = total_latency / len(image_files)

print("\n========== BENCHMARK SUMMARY ==========")
print(f"Model            : MTCNN")
print(f"Images Processed : {len(image_files)}")
print(f"Total Faces      : {total_faces}")
print(f"Average Latency  : {avg_latency:.2f} ms")
print(f"CSV File Saved   : {csv_file}")
print("=======================================")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Loading MTCNN...
MTCNN Loaded Successfully!

Found 7219 images.



 94%|█████████▎| 6755/7219 [55:08<03:54,  1.98it/s]  